# 03 · Consumer Agent

Pablo's agent: the buyer's side. It owns three graph states:

- **INTENT** (LLM + memory): reads the request into a `ParsedQuery`, then fills gaps from the
  saved profile.
- **RANKING** (code): landed cost for every listing, the policy rules, then utility ranking.
- **DONE** (code): saves the profile and the query to SQLite.

Changes from Pablo's version:

- **Persona fix.** The LLM now returns `persona: null` when this message doesn't point to one,
  and memory fills only what wasn't stated. Before, a buyer who said "I don't want to get
  scammed" stayed a saved Budget Parent. The saved profile is no longer shown to the LLM.
- **Consultative questions.** "Is $40 fair for X?" without a budget no longer crashes RANKING.
- **Shared LLM client** (`src/llm.py`) instead of a Gemini client and cache per agent; the
  `llm_parser` hook for tests is unchanged.
- `filled_from_memory` records what came from the profile, so the reply can say so.

In [ ]:
#| default_exp agents.consumer

In [ ]:
%load_ext autoreload
%autoreload 2

import importlib.util
import os
import sys
from pathlib import Path

# Stop early with a clear message if VS Code picked the wrong kernel.
missing = [
    m for m in ("yaml", "dotenv", "langgraph", "pydantic") if not importlib.util.find_spec(m)
]
if missing:
    raise ModuleNotFoundError(
        f"This kernel ({sys.executable}) is not the project environment; missing {missing}. "
        "Run `uv sync` in the repo, then click the kernel name at the top right of the "
        "notebook -> Select Another Kernel -> Python Environments -> .venv (Python 3.14)."
    )

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))  # lets this notebook import the src package

LIVE = False  # True: real Gemini calls with the keys in .env. False: offline, free, no keys.
os.environ["AGENTIC_LLM"] = "primary" if LIVE else "offline"

In [ ]:
#| export
"""Consumer Agent: buyer's assistant, planner, math/ranking executor, and memory owner."""

import re
import time
from collections.abc import Callable
from typing import Any

from pydantic import BaseModel, Field

from src.config import load_config
from src.llm import LLM, make_llm, offline_handler
from src.memory import BuyerMemory
from src.pipeline import SET_HINTS, between, guess_card_name, guess_condition
from src.schemas import Condition, EvaluatedListing, ParsedQuery, Persona, Variant
from src.state import AgentState
from src.tools.landed_cost import compute_landed_cost
from src.tools.policy import evaluate_policy
from src.tools.utility import rank_listings

## INTENT

In [ ]:
#| export
class IntentExtraction(BaseModel):
    """What the LLM reads from the request. Unlike ParsedQuery, every field can be "not
    stated" (None), so code can tell a stated persona from a missing one."""

    card_name: str | None = None
    set_code: str | None = None
    budget: float | None = Field(default=None, ge=0.0)  # Gemini schemas reject gt; 0 = unset
    min_condition: Condition | None = None
    variant: Variant | None = None
    excluded_variants: list[str] = Field(default_factory=list)
    any_printing: bool = False
    persona: Persona | None = None
    is_consultative: bool = False
    asked_price: float | None = Field(default=None, ge=0.0)

In [ ]:
#| export
INTENT_SYSTEM = """Parse the buyer's Pokemon TCG request into structured JSON fields.
Rules:
- card_name: the card with its suffix ("Charizard", "Charizard ex", "Pikachu VMAX").
- set_code: the TCGdex set id if you are sure (e.g. "base1"), otherwise the set name as said
  ("Base Set", "151"). Null if no set is mentioned.
- budget: the most the buyer will spend, in USD. Null if not stated.
- min_condition: NM, LP, MP, HP or DMG only if the buyer sets a minimum ("at least LP").
- variant: normal, holo, reverse or 1st edition if the buyer asks for one;
  excluded_variants for ones they rule out (e.g. ["reverse"]).
- any_printing: true if the buyer says any printing or any set is fine.
- persona: budget_parent (buying for a child or wants the cheapest), serious_collector
  (cares most about condition), cautious_buyer (worried about scams). Null if nothing in
  this message points to one. Do not guess.
- is_consultative: true when the buyer asks for advice or a price check instead of asking
  to buy, e.g. "is $40 fair for a Base Set Charizard?"; asked_price is the price they ask
  about.
Use only what this message says."""

In [ ]:
#| export
def parse_intent(llm: LLM, text: str) -> IntentExtraction:
    prompt = f"Buyer request:\n<request>\n{text}\n</request>"
    return llm.json(tag="intent", system=INTENT_SYSTEM, prompt=prompt, schema=IntentExtraction)

In [ ]:
#| export
def apply_memory(intent: IntentExtraction | ParsedQuery, saved: dict[str, Any]) -> ParsedQuery:
    """Fill what the buyer did not say from the saved profile; a stated value always wins.

    For a ParsedQuery (e.g. from a test's fake parser), a field counts as stated when it was
    set explicitly, so ParsedQuery(card_name="Mew") still picks up the saved persona.
    """
    stated = intent.model_dump(exclude_unset=isinstance(intent, ParsedQuery))
    data = {k: v for k, v in stated.items() if k in ParsedQuery.model_fields and v is not None}
    if not data.get("budget"):
        data.pop("budget", None)  # a budget of 0 means none was given
    filled: list[str] = []
    for field, saved_key in (("budget", "budget"), ("persona", "persona"),
                             ("min_condition", "preferred_condition")):  # fmt: skip
        if data.get(field) is None and saved.get(saved_key) is not None:
            data[field] = float(saved[saved_key]) if field == "budget" else saved[saved_key]
            filled.append(field)
    data["filled_from_memory"] = filled
    return ParsedQuery(**data)

In [ ]:
#| export
def make_intent_node(
    memory: BuyerMemory | None = None,
    config: dict[str, Any] | None = None,
    llm_parser: Callable[[str, dict[str, Any]], IntentExtraction | ParsedQuery] | None = None,
    llm: LLM | None = None,
) -> Callable[[AgentState], dict[str, Any]]:
    """Build the INTENT node (Consumer Agent, LLM + Memory).

    The saved profile is not shown to the LLM: it reads only this message, and code fills
    the gaps from memory afterwards. `llm_parser(text, saved_profile)` replaces the LLM call.
    """
    cfg = config or load_config()
    mem = memory or BuyerMemory()
    model = llm or (None if llm_parser else make_llm(cfg))

    def intent_node(state: AgentState) -> dict[str, Any]:
        t0 = time.perf_counter()
        session_id = state.get("session_id", "default")
        user_input = state.get("user_input", "")
        clarify_answer = state.get("clarify_answer")
        combined_input = (
            f"{user_input}\nClarification answer: {clarify_answer}"
            if clarify_answer
            else user_input
        )

        saved_profile = mem.get_profile(session_id)
        if llm_parser is not None:
            intent = llm_parser(combined_input, saved_profile)
        else:
            intent = parse_intent(model, combined_input)
        parsed = apply_memory(intent, saved_profile)

        timings = dict(state.get("stage_timings", {}))
        timings["INTENT"] = round(time.perf_counter() - t0, 4)
        return {"current_state": "INTENT", "parsed_query": parsed, "stage_timings": timings}

    return intent_node

## RANKING

In [ ]:
#| export
def make_ranking_node(
    config: dict[str, Any] | None = None,
) -> Callable[[AgentState], dict[str, Any]]:
    """Build the RANKING node (Consumer Agent, deterministic Python)."""
    cfg = config or load_config()

    def ranking_node(state: AgentState) -> dict[str, Any]:
        t0 = time.perf_counter()
        parsed = state.get("parsed_query")
        if parsed is None:
            raise ValueError("RANKING requires a parsed_query.")
        # No budget is possible: a consultative question, or the buyer didn't give one even
        # after the one clarifying question. Then the budget rule is skipped (tools/policy.py).

        evaluated = state.get("evaluated_listings", [])
        reflection_complete = bool(state.get("reflection_complete", False))

        default_ship = float(cfg["pricing"]["default_shipping_estimate"])
        tax_rate = float(cfg["pricing"]["default_tax_rate"])
        risk_reject = float(cfg["thresholds"]["risk_reject"])
        risk_warn = float(cfg["thresholds"]["risk_warn"])

        # 1. Compute deterministic landed cost for every evaluated listing
        with_costs: list[EvaluatedListing] = []
        for item in evaluated:
            lc = compute_landed_cost(
                price=item.listing.price,
                shipping=item.listing.shipping,
                delivery_type=item.listing.delivery_type,
                default_shipping_estimate=default_ship,
                tax_rate=tax_rate,
            )
            with_costs.append(item.model_copy(update={"landed_cost": lc}))

        # 2. Enforce Platform policy rules (budget cap, risk ceiling, reflection gate)
        policy_res = evaluate_policy(
            candidates=with_costs,
            budget=parsed.budget,
            risk_reject_threshold=risk_reject,
            risk_warn_threshold=risk_warn,
            reflection_complete=reflection_complete,
        )

        # 3. Rank surviving listings with persona-weighted utility
        ranked = rank_listings(
            listings=policy_res.passing,
            budget=parsed.budget,
            persona=parsed.persona,
            config=cfg,
        )

        timings = dict(state.get("stage_timings", {}))
        timings["RANKING"] = round(time.perf_counter() - t0, 4)

        return {
            "current_state": "RANKING",
            "ranked_listings": ranked,
            "rejected_listings": policy_res.rejected,
            "closest_rejected": policy_res.closest_rejected,
            "stage_timings": timings,
        }

    return ranking_node

## DONE

In [ ]:
#| export
def make_done_node(
    memory: BuyerMemory | None = None,
) -> Callable[[AgentState], dict[str, Any]]:
    """Build the DONE node (Consumer Agent, persists session memory to SQLite)."""
    mem = memory or BuyerMemory()

    def done_node(state: AgentState) -> dict[str, Any]:
        t0 = time.perf_counter()
        session_id = state.get("session_id", "default")
        parsed = state.get("parsed_query")
        ranked = state.get("ranked_listings", [])
        top_pick_id = ranked[0].listing.listing_id if ranked else None

        if parsed is not None:
            mem.save_profile(
                session_id=session_id,
                budget=parsed.budget,
                persona=parsed.persona,
                preferred_set=parsed.set_code,
                preferred_condition=parsed.min_condition,
            )
            mem.log_query(
                session_id=session_id,
                raw_query=state.get("user_input", ""),
                parsed_dict=parsed.model_dump(),
                top_pick_id=top_pick_id,
            )

        timings = dict(state.get("stage_timings", {}))
        timings["DONE"] = round(time.perf_counter() - t0, 4)

        return {
            "current_state": "DONE",
            "stage_timings": timings,
        }

    return done_node

## Offline stand-in for the intent call

In [ ]:
#| export
@offline_handler("intent")
def offline_intent(_system: str, prompt: str) -> dict:
    """Rule-based stand-in for the intent call (offline mode)."""
    text = between(prompt, "request")
    low = text.lower()
    out: dict[str, Any] = {"card_name": guess_card_name(text), "excluded_variants": []}
    out["set_code"] = next((name for p, name in SET_HINTS if re.search(p, low)), None)
    if re.search(r"\b(fair|worth|good price|good deal|reasonable)\b", low):
        out["is_consultative"] = True
        pm = re.search(r"\$\s*(\d+(?:\.\d+)?)", low)
        out["asked_price"] = float(pm.group(1)) if pm else None
        out["min_condition"] = guess_condition(text)
    else:
        m = re.search(
            r"(?:under|below|budget(?: is)?|max(?:imum)?|up to|spend|have)\s*\$?\s*(\d+(?:\.\d+)?)",
            low,
        )
        m = (
            m
            or re.search(r"\$\s*(\d+(?:\.\d+)?)", low)
            or re.search(r"(\d+(?:\.\d+)?)\s*(?:dollars|bucks)", low)
        )
        if m:
            out["budget"] = float(m.group(1))
    if re.search(r"\b(kid|son|daughter|child|cheap|cheapest)\b", low):
        out["persona"] = "budget_parent"
    elif re.search(r"\b(scam\w*|safe\w*|legit|trust\w*|fake)\b", low):
        out["persona"] = "cautious_buyer"
    elif re.search(r"\b(condition|collector|collection|mint)\b", low):
        out["persona"] = "serious_collector"
    if re.search(r"\bany\b", low):
        out["any_printing"] = True
    if re.search(r"(not|no|without) (a )?reverse", low):
        out["excluded_variants"].append("reverse")
    elif "reverse" in low:
        out["variant"] = "reverse"
    if "1st edition" in low or "first edition" in low:
        out["variant"] = "1st edition"
    at_least = re.search(r"at least (nm|lp|mp|hp|near mint|lightly played)", low)
    if at_least:
        out["min_condition"] = guess_condition(at_least.group(1))
    return out

# Try it

In [ ]:
import json
import tempfile

import pandas as pd

from src.config import load_config, repo_path
from src.llm import make_llm
from src.memory import BuyerMemory

cfg = load_config()
llm = make_llm(cfg)
queries = json.loads(repo_path("data/samples/sample_queries.json").read_text(encoding="utf-8"))
rows = [
    {"request": q["request"], **parse_intent(llm, q["request"]).model_dump(exclude_defaults=True)}
    for q in queries
]
pd.DataFrame(rows)

Memory fills only what this message leaves out; a stated persona always wins:

In [ ]:
mem = BuyerMemory(Path(tempfile.mkdtemp()) / "demo.sqlite")
mem.save_profile("demo", budget=50, persona="budget_parent")
intent = make_intent_node(memory=mem, config=cfg, llm=llm)
for text in ["Base Set Mewtwo please", "Base Set Mewtwo, I really don't want to get scammed"]:
    q = intent({"session_id": "demo", "user_input": text})["parsed_query"]
    print(f"{text!r}\n  persona={q.persona} budget={q.budget} from memory={q.filled_from_memory}")

## Export

Run this after you change any `#| export` cell. It rewrites the generated module in `src/`,
so the other notebooks, the graph and the tests use your new version.

In [ ]:
from src.nbexport import export_all

changed = export_all()